# Photon Mosaic: synthetic data walkthrough

Generates a synthetic imaging movie with known ground-truth fluorescence, then runs it
through the full analysis pipeline: fluorescence extraction, dF/F, and deconvolution.

In [ ]:
import numpy as np
import spikeinterface.widgets as sw

import photon_mosaic as pm
import photon_mosaic.widgets as pw

%matplotlib widget

## Generate synthetic imaging data

In [ ]:
rois, imaging, ground_truth = pm.generate_imaging_with_rois(
    num_frames=10000, bleaching_time=600.0, noise_std="poisson", seed=0
)

In [ ]:
rois

In [ ]:
imaging

In [ ]:
pw.plot_imaging_series(imaging, backend="ipywidgets")

In [ ]:
pw.plot_rois(rois, backend="ipywidgets", width_cm=20)

## Create the ROI analyzer

In [ ]:
analyzer = pm.create_roi_analyzer(rois, imaging)

## Extract fluorescence

In [ ]:
fluorescence_ext = analyzer.compute("fluorescence", n_jobs=4)

In [ ]:
fluorescence = fluorescence_ext.get_data(outputs="recording")

In [ ]:
sw.plot_traces(fluorescence, backend="ipywidgets", time_range=[0, 30])

## Compute dF/F

In [ ]:
dff_ext = analyzer.compute("df_over_f", n_jobs=4, method="percentile")
df_over_f = dff_ext.get_data(outputs="recording")
sw.plot_traces(df_over_f, backend="ipywidgets", time_range=[30, 60])

In [ ]:
dff_ext_maximin = analyzer.compute("df_over_f", n_jobs=4, method="maximin")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

t = np.arange(fluorescence_ext.get_data().shape[0]) / imaging.sampling_frequency
fig, ax = plt.subplots(3, 1, figsize=(10, 4), sharex=True)
for i in rois.roi_ids[:3]:
    ax[0].plot(t, fluorescence_ext.get_data()[:,i], c=f"C{i}")
    ax[1].plot(t, dff1:=dff_ext.get_data()[:,i], c=f"C{i}", alpha=0.5)
    ax[1].plot(t, dff2:=dff_ext_maximin.get_data()[:,i], c=f"C{i}", ls="--")
    ax[2].plot(t, dff1-dff2, c=f"C{i}")
ax[0].set_ylabel("fluorescence")
ax[1].set_ylabel("dF/F")
ax[2].set_ylabel("percentile - maximin")
ax[2].set_xlabel("time (s)")

roi_handles = [Line2D([0], [0], color=f"C{i}", label=f"ROI {i}") for i in rois.roi_ids[:3]]
for a in ax:
    a.legend(handles=roi_handles, loc="upper right", fontsize=8)

style_handles = [
    Line2D([0], [0], color="k", alpha=0.5, label="percentile"),
    Line2D([0], [0], color="k", ls="--", label="maximin"),
]
style_legend = ax[1].legend(handles=style_handles, loc="upper left", fontsize=8)
ax[1].add_artist(style_legend)
ax[1].legend(handles=roi_handles, loc="upper right", fontsize=8)
fig.tight_layout()

## Deconvolution

In [ ]:
deconv_ext = analyzer.compute("deconvolution", n_jobs=4)

In [ ]:
deconvolved = deconv_ext.get_data(outputs="recording")
sw.plot_traces(deconvolved, backend="ipywidgets", time_range=[30, 60])

**Note:** the cell below checks which of the plotted ROIs share pixels with other ROIs.
Unmixed crosstalk between overlapping ROIs can cause small false transients/events in the
traces shown further down.

In [ ]:
masks = rois.get_roi_image_masks() > 0
for i in rois.roi_ids[:3]:
    overlapping = [int(j) for j in rois.roi_ids if j != i and np.any(masks[i] & masks[j])]
    print(f"ROI {i} overlaps with: {overlapping}")

**Note:** inferred dF/F is systematically smaller than ground truth in the plot below. The
video's `background` (neuropil, out-of-focus light, dark counts) doesn't scale with each ROI's
F0/expression level and isn't subtracted here, which attenuates recovered dF/F (see
`generate_imaging_with_rois`'s `background` docs and `FluorescenceNode`'s `neuropil` argument
to correct for this).

In [ ]:
t = np.arange(dff_ext.get_data().shape[0]) / imaging.sampling_frequency
fig, ax = plt.subplots(3, 1, figsize=(10, 4), sharex=True)
for i in rois.roi_ids[:3]:
    ax[0].plot(t, dff_ext.get_data()[:, i], c=f"C{i}", lw=2.5, alpha=0.5)
    ax[0].plot(t, ground_truth.clean_traces[:, i], c=f"C{i}", ls="--")
    ax[1].plot(t, deconv_ext.data["denoised"][:, i], c=f"C{i}", lw=2.5, alpha=0.5)
    ax[1].plot(t, ground_truth.clean_traces[:, i], c=f"C{i}", ls="--")
    ax[2].plot(t, deconv_ext.get_data()[:, i], c=f"C{i}", lw=2.5, alpha=0.5)
    ax[2].plot(t, ground_truth.spikes[:, i], c=f"C{i}", ls="--")
ax[0].set_ylabel("dF/F")
ax[1].set_ylabel("denoised")
ax[2].set_ylabel("deconvolved")
ax[2].set_xlabel("time (s)")

style_handles = [
    Line2D([0], [0], color="k", lw=2.5, alpha=0.5, label="inferred"),
    Line2D([0], [0], color="k", ls="--", label="ground truth"),
]
roi_handles = [Line2D([0], [0], color=f"C{i}", label=f"ROI {i}") for i in rois.roi_ids[:3]]
for a in ax:
    style_legend = a.legend(handles=style_handles, loc="upper left", fontsize=8)
    a.add_artist(style_legend)
    a.legend(handles=roi_handles, loc="upper right", fontsize=8)
fig.tight_layout()

**Note:** the plot below repeats the comparison above, but with each ROI's inferred trace
rescaled to its ground truth via least-squares regression without an intercept
(`scale = argmin_a ||a * inferred - ground_truth||^2 = (inferred . ground_truth) / (inferred . inferred)`),
to check whether the shape recovery is faithful once the background-induced amplitude
attenuation noted above is corrected for.

In [ ]:
def fit_scale(inferred, ground_truth):
    return (inferred * ground_truth).sum(axis=0) / (inferred * inferred).sum(axis=0)


dff_scale = fit_scale(dff_ext.get_data()[:, :3], ground_truth.clean_traces[:, :3])
denoised_scale = fit_scale(deconv_ext.data["denoised"][:, :3], ground_truth.clean_traces[:, :3])
deconvolved_scale = fit_scale(deconv_ext.get_data()[:, :3], ground_truth.spikes[:, :3])
print("dF/F scale factors:", dff_scale)
print("denoised scale factors:", denoised_scale)
print("deconvolved scale factors:", deconvolved_scale)

fig, ax = plt.subplots(3, 1, figsize=(10, 4), sharex=True)
for i in rois.roi_ids[:3]:
    ax[0].plot(t, dff_scale[i] * dff_ext.get_data()[:, i], c=f"C{i}", lw=2.5, alpha=0.5)
    ax[0].plot(t, ground_truth.clean_traces[:, i], c=f"C{i}", ls="--")
    ax[1].plot(t, denoised_scale[i] * deconv_ext.data["denoised"][:, i], c=f"C{i}", lw=2.5, alpha=0.5)
    ax[1].plot(t, ground_truth.clean_traces[:, i], c=f"C{i}", ls="--")
    ax[2].plot(t, deconvolved_scale[i] * deconv_ext.get_data()[:, i], c=f"C{i}", lw=2.5, alpha=0.5)
    ax[2].plot(t, ground_truth.spikes[:, i], c=f"C{i}", ls="--")
ax[0].set_ylabel("dF/F")
ax[1].set_ylabel("denoised")
ax[2].set_ylabel("deconvolved")
ax[2].set_xlabel("time (s)")

style_handles = [
    Line2D([0], [0], color="k", lw=2.5, alpha=0.5, label="inferred (rescaled)"),
    Line2D([0], [0], color="k", ls="--", label="ground truth"),
]
roi_handles = [Line2D([0], [0], color=f"C{i}", label=f"ROI {i}") for i in rois.roi_ids[:3]]
for a in ax:
    style_legend = a.legend(handles=style_handles, loc="upper left", fontsize=8)
    a.add_artist(style_legend)
    a.legend(handles=roi_handles, loc="upper right", fontsize=8)
fig.tight_layout()